# External Data fusion
`Build the contextual environment dataset completely isolated from the modeling logic`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
BASE_DIR = Path().resolve().parent

In [ ]:

#* time series dataset (to extract timestamps)
time_series_path = BASE_DIR / 'Dataset' / 'ai4i2020_time_series.csv'
internal_df = pd.read_csv(time_series_path)

timestamps = pd.to_datetime(internal_df['Timestamp']).unique()
total_rows = len(timestamps)
print(f'extracted temporal anchor of {total_rows} minute-by-minute operational cycles')

In [ ]:

#? extract time components
hours = pd.DatetimeIndex(timestamps).hour
days = pd.DatetimeIndex(timestamps).day

#todo set random seed to prevent 'random randomness'!
np.random.seed(42)

## External feature generation
`8 contextual environment signals derived purely from the extracted hours / days / total_rows anchors`

In [ ]:
#* External feature generation — 8 contextual environment signals derived from hours / days / total_rows

# 1. ambient_temp (Ambient Temperature, Kelvin)
ambient_temp = 298.15 + 6.0 * np.cos(2 * np.pi * (hours - 14) / 24) + np.random.normal(0, 0.4, total_rows)

# 2. ambient_humidity (Ambient Humidity, %)
ambient_humidity = 60.0 - 12.0 * np.cos(2 * np.pi * (hours - 14) / 24) + np.random.normal(0, 1.0, total_rows)

# 3. atmospheric_pressure (hPa)
atmospheric_pressure = 1013.25 + 3.0 * np.sin(2 * np.pi * days / 5) + np.random.normal(0, 0.5, total_rows)

# 4. grid_voltage_fluctuation (V)
grid_voltage_fluctuation = np.random.normal(0, 1.5, total_rows) + np.where(
    np.random.rand(total_rows) > 0.98,
    np.random.uniform(-5.0, -2.0, total_rows),
    0
)

# 5. factory_load_density (Floor Congestion, 0-1)
factory_load_density = np.where(
    (hours >= 8) & (hours <= 20),
    np.random.uniform(0.75, 0.95, total_rows),
    np.random.uniform(0.30, 0.55, total_rows)
)

# 6. operator_skill_proxy (Shift Experience Level)
operator_skill_proxy = np.where(
    (hours >= 8) & (hours < 16), 1,
    np.where((hours >= 16) & (hours < 24), 2, 3)
)

# 7. particulate_matter_pm10 (Airborne Dust)
particulate_matter_pm10 = 35.0 + (factory_load_density * 25.0) + np.random.exponential(5.0, total_rows)

# 8. ambient_vibration_noise (Background Decibels)
ambient_vibration_noise = 45.0 + (factory_load_density * 15.0) + np.random.normal(0, 1.2, total_rows)

### Assemble the external environment dataset

In [ ]:
#Created a Dataframe for the features
external_env_df = pd.DataFrame({
    'Timestamp': timestamps,
    'ambient_temp': ambient_temp,
    'ambient_humidity': ambient_humidity,
    'atmospheric_pressure': atmospheric_pressure,
    'grid_voltage_fluctuation': grid_voltage_fluctuation,
    'factory_load_density': factory_load_density,
    'operator_skill_proxy': operator_skill_proxy,
    'particulate_matter_pm10': particulate_matter_pm10,
    'ambient_vibration_noise': ambient_vibration_noise,
})

external_env_df.head()